# 04 — Fine-tune PP-OCRv5 reader without Drive mount

Upload the four frozen v001 files from `E:\backup-tramcan\colab_upload_v001` when prompted.
Run one code cell at a time through the smoke test. Do not use Run all: the full
training cell follows the smoke test. Everything under `/content` disappears
when the Colab runtime ends, so download a checkpoint archive before leaving.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
from google.colab import files

REPO_URL = 'https://github.com/huybitvvt/scale-ocr.git'
PROJECT_COMMIT = 'b0902866d0a04a015a9b0edf2f75f106d1851ef1'
VERSION = 'v001'
LOCAL_ROOT = Path('/content/scale-data')
RUN_ROOT = Path('/content/scale-runs')
REPO_ROOT = Path('/content/scale-ocr-code')
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', PROJECT_COMMIT], check=True)
actual_commit = subprocess.check_output(
    ['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == PROJECT_COMMIT
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_ROOT)], check=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Code commit:', actual_commit)
print('Disk GiB free:', round(shutil.disk_usage('/content').free / 2**30, 2))
assert shutil.which('nvidia-smi'), 'GPU not enabled: Runtime > Change runtime type > T4 GPU, then reconnect'
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
# Select all four files from E:\backup-tramcan\colab_upload_v001.
os.chdir('/content')
required = ('dataset.zip', 'dataset.zip.sha256',
            'labels_reviewed_v001.zip', 'labels_reviewed_v001.zip.sha256')
if not all((Path('/content') / name).is_file() for name in required):
    uploaded = files.upload()
    del uploaded
missing = [name for name in required if not (Path('/content') / name).is_file()]
assert not missing, f'Missing uploaded files: {missing}'

def sha256_stream(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

local_bundle = Path('/content/dataset.zip')
expected = '0eebc32cb89e8d264d0450292e57aeeb81388efe0d9d15848ab3d7489b91870b'
labels_expected = 'c2f0a77dacad33face7c74a5180fce5ae79498514e38d89730de4e22c94f78b6'
assert Path('/content/dataset.zip.sha256').read_text(encoding='ascii').split()[0].lower() == expected
assert Path('/content/labels_reviewed_v001.zip.sha256').read_text(encoding='ascii').split()[0].lower() == labels_expected
actual = sha256_stream(local_bundle)
assert actual == expected, f'Dataset SHA-256 mismatch: {actual}'
assert sha256_stream(Path('/content/labels_reviewed_v001.zip')) == labels_expected
prepared = LOCAL_ROOT / 'prepared' / VERSION
if not (prepared / 'dataset_card.json').is_file():
    prepared.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_bundle) as archive:
        for item in archive.infolist():
            target = (prepared / item.filename).resolve()
            assert target.is_relative_to(prepared.resolve()), 'Unsafe ZIP path'
        assert archive.testzip() is None, 'Corrupt dataset ZIP'
        archive.extractall(prepared)
card = json.loads((prepared / 'dataset_card.json').read_text(encoding='utf-8'))
assert card['counts'].get('recognizer_crops_train', 0) > 0
print('Dataset SHA-256:', actual)
print('Reviewed labels SHA-256:', labels_expected)
print('Counts:', card['counts'])


In [ ]:
# Ví dụ wheel CUDA 12.6 theo hướng dẫn Paddle; sửa theo runtime hiện được cấp.
PADDLE_WHEEL_INDEX = 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'
PADDLE_WHEEL = 'paddlepaddle-gpu==3.3.0'
subprocess.run([sys.executable, '-m', 'pip', 'install', PADDLE_WHEEL,
                '-i', PADDLE_WHEEL_INDEX], check=True)
PADDLE_ROOT = Path('/content/PaddleOCR')
PADDLE_REF = 'release/3.5'
PADDLE_COMMIT = '9d5e7663a3d7035456538435fae82a1762cd3529'
if not PADDLE_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', PADDLE_REF,
        'https://github.com/PaddlePaddle/PaddleOCR.git', str(PADDLE_ROOT)], check=True)
current_paddle_commit = subprocess.check_output(
    ['git', '-C', str(PADDLE_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
if current_paddle_commit != PADDLE_COMMIT:
    subprocess.run(['git', '-C', str(PADDLE_ROOT), 'fetch', '--depth', '1',
                    'origin', PADDLE_COMMIT], check=True)
    subprocess.run(['git', '-C', str(PADDLE_ROOT), 'checkout', '--detach',
                    PADDLE_COMMIT], check=True)
paddle_commit = subprocess.check_output(
    ['git', '-C', str(PADDLE_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
assert paddle_commit == PADDLE_COMMIT
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(PADDLE_ROOT / 'requirements.txt')], check=True)
import paddle, yaml
assert paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0
paddle.set_device('gpu:0')
paddle.utils.run_check()
print('PaddleOCR commit:', paddle_commit)


In [ ]:
from urllib.request import urlretrieve
from datetime import datetime, timezone
pretrained = LOCAL_ROOT / 'pretrained' / 'PP-OCRv5_server_rec_pretrained.pdparams'
pretrained.parent.mkdir(parents=True, exist_ok=True)
if not pretrained.exists():
    urlretrieve('https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/'
                'PP-OCRv5_server_rec_pretrained.pdparams', pretrained)
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
rec_root = prepared / 'recognizer'
for split in ('train', 'val'):
    assert (rec_root / (split + '.txt')).stat().st_size > 0, f'Thiếu nhãn {split}'
base_config = PADDLE_ROOT / 'configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml'
cfg = yaml.safe_load(base_config.read_text(encoding='utf-8'))
REC_RUN_ID = 'rec-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
rec_run = RUN_ROOT / REC_RUN_ID
rec_run.mkdir(parents=True, exist_ok=False)
cfg['Global'].update({'use_gpu': True, 'distributed': False, 'epoch_num': 50,
    'pretrained_model': str(pretrained), 'checkpoints': None,
    'save_model_dir': str(rec_run / 'checkpoints'), 'save_epoch_step': 5,
    'eval_batch_step': [0, 100]})
cfg['Optimizer']['lr']['learning_rate'] = 0.0001
for section, label in [('Train', 'train.txt'), ('Eval', 'val.txt')]:
    cfg[section]['dataset']['data_dir'] = str(rec_root)
    cfg[section]['dataset']['label_file_list'] = [str(rec_root / label)]
    cfg[section]['loader']['num_workers'] = 2
    cfg[section]['loader']['batch_size_per_card'] = 16
cfg['Train']['sampler']['first_bs'] = 16
cfg['Train']['sampler']['fix_bs'] = True
cfg['Train']['sampler']['scales'] = [[320, 48]]
cfg['Train']['dataset']['transforms'] = [t for t in cfg['Train']['dataset']['transforms']
    if 'RecAug' not in t]
rec_config = rec_run / 'recognizer.yaml'
rec_config.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
(rec_run / 'run_manifest.json').write_text(json.dumps({
    'project_commit': PROJECT_COMMIT, 'dataset_sha256': actual,
    'dataset_version': VERSION, 'paddleocr_commit': paddle_commit,
    'pretrained_sha256': sha256(pretrained), 'paddle': paddle.__version__,
    'config_sha256': sha256(rec_config), 'seed': 42,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print('Run:', REC_RUN_ID)


## Smoke test trước run chính

    Cell này train 1 epoch trên 64 crop train và 16 crop val. Checkpoint thử nghiệm nằm trong `/content`; chỉ chạy cell train chính nếu smoke test kết thúc thành công và log cho thấy model nạp pretrained đúng.


In [ ]:
import copy

smoke_dir = LOCAL_ROOT / 'smoke' / REC_RUN_ID
smoke_dir.mkdir(parents=True, exist_ok=True)
train_lines = (rec_root / 'train.txt').read_text(encoding='utf-8').splitlines(keepends=True)
val_lines = (rec_root / 'val.txt').read_text(encoding='utf-8').splitlines(keepends=True)
assert len(train_lines) >= 64 and len(val_lines) >= 16
smoke_train = smoke_dir / 'train.txt'
smoke_val = smoke_dir / 'val.txt'
smoke_train.write_text(''.join(train_lines[:64]), encoding='utf-8')
smoke_val.write_text(''.join(val_lines[:16]), encoding='utf-8')
smoke_cfg = copy.deepcopy(cfg)
smoke_cfg['Global'].update({
    'epoch_num': 1, 'save_epoch_step': 1,
    'eval_batch_step': [0, 4],
    'save_model_dir': str(smoke_dir / 'checkpoints'),
})
smoke_cfg['Train']['dataset']['label_file_list'] = [str(smoke_train)]
smoke_cfg['Eval']['dataset']['label_file_list'] = [str(smoke_val)]
smoke_cfg['Train']['sampler']['first_bs'] = 8
smoke_cfg['Train']['loader']['batch_size_per_card'] = 8
smoke_cfg['Eval']['loader']['batch_size_per_card'] = 8
smoke_config = smoke_dir / 'recognizer_smoke.yaml'
smoke_config.write_text(yaml.safe_dump(smoke_cfg, sort_keys=False), encoding='utf-8')
subprocess.run([sys.executable, 'tools/train.py', '-c', str(smoke_config)],
               cwd=PADDLE_ROOT, check=True)
print('Smoke test completed:', smoke_dir)


## Full reader training

Run only after the one-epoch smoke test succeeds. The checkpoint is saved under
`/content/scale-runs`, including `latest` at every epoch. If training stops,
run the backup cell below while the same runtime is still alive.


In [ ]:
subprocess.run([sys.executable, 'tools/train.py', '-c', str(rec_config)],
               cwd=PADDLE_ROOT, check=True)


## Download a resumable checkpoint

Run this cell after training, or after stopping a run that has completed at
least one epoch. Save the downloaded ZIP on your PC and then upload it to Drive
using the browser. Confirm the browser download has actually finished before
disconnecting the Colab runtime.


In [ ]:
ckpt_dir = rec_run / 'checkpoints'
latest = [ckpt_dir / ('latest' + ext) for ext in ('.pdparams', '.pdopt', '.states')]
assert all(path.is_file() for path in latest), 'No complete latest checkpoint yet'
selected = [rec_run / 'recognizer.yaml', rec_run / 'run_manifest.json', *latest]
selected += [path for path in ckpt_dir.glob('best_accuracy.*') if path.is_file()]
backup_zip = Path('/content') / f'{REC_RUN_ID}_checkpoint.zip'
with zipfile.ZipFile(backup_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for path in selected:
        archive.write(path, path.relative_to(RUN_ROOT))
with zipfile.ZipFile(backup_zip) as archive:
    assert archive.testzip() is None, 'Checkpoint ZIP is corrupt'
print('Checkpoint archive:', backup_zip, 'bytes:', backup_zip.stat().st_size)
print('SHA-256:', sha256_stream(backup_zip))
files.download(str(backup_zip))


## Resume in a new runtime

Run the first four code cells again to restore code, dataset, PaddleOCR and
the pretrained model. Skip the smoke and full-train cells. Then run this cell
and select the checkpoint ZIP downloaded earlier. Download a new checkpoint
after the resumed training ends.


In [ ]:
os.chdir('/content')
uploaded = files.upload()
archive_names = list(uploaded)
del uploaded
assert len(archive_names) == 1 and archive_names[0].endswith('_checkpoint.zip')
resume_zip = Path('/content') / archive_names[0]
with zipfile.ZipFile(resume_zip) as archive:
    names = [item.filename for item in archive.infolist()]
    run_names = {Path(name).parts[0] for name in names if Path(name).parts}
    assert len(run_names) == 1, run_names
    resume_id = run_names.pop()
    assert resume_id.startswith('rec-')
    for item in archive.infolist():
        target = (RUN_ROOT / item.filename).resolve()
        assert target.is_relative_to(RUN_ROOT.resolve()), 'Unsafe checkpoint ZIP path'
    assert archive.testzip() is None, 'Corrupt checkpoint ZIP'
    archive.extractall(RUN_ROOT)
rec_run = RUN_ROOT / resume_id
old = json.loads((rec_run / 'run_manifest.json').read_text(encoding='utf-8'))
assert old['project_commit'] == PROJECT_COMMIT and old['dataset_sha256'] == actual
assert old['paddleocr_commit'] == paddle_commit
rec_config = rec_run / 'recognizer.yaml'
assert sha256_stream(rec_config) == old['config_sha256']
prefix = rec_run / 'checkpoints' / 'latest'
assert all(prefix.with_suffix(ext).is_file() for ext in ('.pdparams', '.pdopt', '.states'))
REC_RUN_ID = resume_id
subprocess.run([sys.executable, 'tools/train.py', '-c', str(rec_config),
                '-o', 'Global.checkpoints=' + str(prefix)],
               cwd=PADDLE_ROOT, check=True)
